# Channel Projection Analysis

This notebook analyzes the relationship between semantic channel descriptions and their point cloud (video title embeddings) topology. It projects both into 2D space, aligns them, and creates a transition animation.

## 1) Setup & Dependencies

Install requirements, mount Google Drive, and import necessary libraries.

This cell defines the `load_centroids` function, which reads the 20D video title embeddings from a CSV file and calculates the mean embedding for each channel. If the file is not found, it generates random dummy data to allow the notebook to run in a headless environment. The output is a DataFrame containing the 20D centroids indexed by channel name.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from pathlib import Path
from scipy.spatial import procrustes
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer

# Install dependencies if needed (Colab already has most)
# !pip install -q pandas numpy scipy sentence-transformers matplotlib

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    print('Not running in Colab. Falling back to local execution.')
    IN_COLAB = False

## 2) Configuration & Paths

Define input data paths and output locations.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
GRAPHIKO_ROOT = DRIVE_ROOT / 'Graphiko'

VIDEO_EMBEDDINGS_CSV = (
    GRAPHIKO_ROOT
    / 'exports/video_embeddings_reduced/latest/business_cluster_video_embeddings_reduced_20d.csv'
)

MERGED_DESCRIPTIONS_JSON = (
    GRAPHIKO_ROOT
    / 'research/merged_channel_descriptions/latest/all_channel_descriptions_by_source.json'
)

ANIMATION_OUTPUT_PATH = GRAPHIKO_ROOT / 'research/projection_transition.mp4'

# Fallback for local testing
if not VIDEO_EMBEDDINGS_CSV.exists():
    VIDEO_EMBEDDINGS_CSV = Path('business_cluster_video_embeddings_reduced_20d.csv')
if not MERGED_DESCRIPTIONS_JSON.exists():
    MERGED_DESCRIPTIONS_JSON = Path('all_channel_descriptions_by_source.json')

## 3) Load Data and Compute Centroids

Load the 20D video title embeddings and calculate the mean centroid for each channel.

In [ ]:
def load_centroids(path):
    if not path.exists():
        # Dummy data for headless execution if file missing
        print(f'Warning: {path} not found. Using dummy data.')
        channels = [f'Channel {i}' for i in range(10)]
        centroids = np.random.randn(10, 20)
        return pd.DataFrame(centroids, index=channels, columns=[f'embedding_reduced_{i:02d}' for i in range(1, 21)])

    df = pd.read_csv(path)
    emb_cols = [f'embedding_reduced_{i:02d}' for i in range(1, 21)]
    centroids = df.groupby('channel_name')[emb_cols].mean()
    return centroids

title_centroids = load_centroids(VIDEO_EMBEDDINGS_CSV)
channel_names = title_centroids.index.tolist()
print(f'Loaded centroids for {len(channel_names)} channels.')

## 4) Load and Encode Descriptions

This section handles the loading of channel descriptions generated by the 'Jules' source and their encoding into semantic vectors.

This cell defines `load_and_encode_descriptions`, which reads the merged channel descriptions from a JSON file, extracts the text associated with the 'Jules' persona, and uses the `all-MiniLM-L6-v2` SentenceTransformer model to create high-dimensional embeddings. It includes fallback dummy data generation and expected output is a numpy array of embeddings.

In [ ]:
def load_and_encode_descriptions(path, channels):
    if not path.exists():
        print(f'Warning: {path} not found. Using dummy embeddings.')
        return np.random.randn(len(channels), 384) # MiniLM dim

    with open(path, 'r') as f:
        data = json.load(f)

    descriptions = []
    for ch in channels:
        # Extract description from Jules source, removing prefix if present
        desc = data['channels'].get(ch, {}).get('Jules', '')
        if desc.startswith('Jules: '):
            desc = desc[len('Jules: '):]
        descriptions.append(desc)

    # Use the genai SDK if available for other tasks, but for encoding MiniLM is standard here
    model = SentenceTransformer('all-MiniLM-L6-v2')
    return model.encode(descriptions)

desc_embeddings = load_and_encode_descriptions(MERGED_DESCRIPTIONS_JSON, channel_names)
print(f'Encoded {len(desc_embeddings)} descriptions.')

## 5) Projection and Alignment

 dimensionality reduction is performed on both the video title centroids and the description embeddings to project them into a shared 2D space for visualization.

This cell utilizes Principal Component Analysis (PCA) to reduce the dimensionality of both data sets to 2 components. Following this, Procrustes analysis is applied to align the description-based coordinates to the title-centroid coordinates. This alignment step is crucial for minimizing the influence of arbitrary rotations and scaling, thereby highlighting true semantic shifts in the subsequent animation. The function returns two sets of aligned 2D coordinates.

In [ ]:
def project_and_align(titles, descriptions):
    pca_t = PCA(n_components=2)
    pos_titles = pca_t.fit_transform(titles)

    pca_d = PCA(n_components=2)
    pos_descriptions = pca_d.fit_transform(descriptions)

    # Procrustes alignment: maps pos_descriptions to pos_titles
    mt, md, disparity = procrustes(pos_titles, pos_descriptions)
    # mt and md are the aligned versions of pos_titles and pos_descriptions
    # (normalized, rotated, scaled)

    return mt, md

coords1, coords2 = project_and_align(title_centroids.values, desc_embeddings)
print('Projections and alignment complete.')

## 6) Create Animation

The final step is to generate an animation that visually demonstrates the transition between the point cloud topology and the description-induced topology.

This cell sets up a Matplotlib animation using `FuncAnimation`. It maps each channel to a unique and consistent color. The animation interpolates node positions between the two aligned projections. The resulting sequence is exported as an MP4 video file to the specified path in Google Drive. Expected output is a saved video file.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Use a consistent colormap for channels
colors = plt.cm.rainbow(np.linspace(0, 1, len(channel_names)))

scat = ax.scatter(coords1[:, 0], coords1[:, 1], c=colors, s=100, edgecolors='k')
texts = [ax.text(coords1[i, 0], coords1[i, 1], name, fontsize=8) 
         for i, name in enumerate(channel_names)]

def update(frame):
    # Interpolate between coords1 and coords2
    t = frame / 100.0
    curr_coords = (1 - t) * coords1 + t * coords2
    
    scat.set_offsets(curr_coords)
    for i, text in enumerate(texts):
        text.set_position((curr_coords[i, 0], curr_coords[i, 1]))
    
    ax.set_title(f'Transition: Title Centroids -> Jules Descriptions ({int(t*100)}%)')

    return scat, *texts

ani = animation.FuncAnimation(fig, update, frames=np.arange(0, 101), interval=50, blit=True)

ax.set_xlim(min(coords1[:,0].min(), coords2[:,0].min()) - 0.1, 
            max(coords1[:,0].max(), coords2[:,0].max()) + 0.1)
ax.set_ylim(min(coords1[:,1].min(), coords2[:,1].min()) - 0.1, 
            max(coords1[:,1].max(), coords2[:,1].max()) + 0.1)

try:
    print(f'Saving animation to {ANIMATION_OUTPUT_PATH}...')
    ani.save(str(ANIMATION_OUTPUT_PATH), writer='ffmpeg')
    print('Animation saved successfully.')
except Exception as e:
    print(f'Error saving animation: {e}')

plt.close(fig)